# Smart Agarbatti — Final Integrated Quality Inspection Pipeline

## YOLOv8 + OpenCV + Drying Status + Rule-Based Decision Engine

This notebook is designed around the actual prototype:

```text
SOLAR PANEL → BATTERY → ESP32 → DRYING CHAMBER → CAMERA
                                  ↑
                           TEMP / HUMIDITY
                                  ↓
                           DRYING STATUS

CAMERA IMAGE
     ↓
YOLOv8
  Detects:
  • Ready
  • Broken
  • Bent
     ↓
OpenCV on YOLO ROI
  Measures:
  • Width / height in pixels
  • Aspect ratio
  • Colour / brightness
     ↓
Drying status from ESP32 (optional input)
     +
YOLO result
     +
OpenCV measurements
     ↓
RULE-BASED DECISION ENGINE
     ↓
GRADE A / GRADE C / REJECT / MANUAL CHECK

# 1. Install libraries

Run this once in Google Colab/Jupyter.

In [ ]:
%pip install -q ultralytics opencv-python matplotlib pandas pyyaml

# 2. Imports and project configuration

**Important:** YOLO is used for visual recognition. OpenCV is used for deterministic image measurements. The decision engine is ordinary Python rules — it is **not another AI model**.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

print("Libraries imported successfully.")

# 3. YOLO classes

For the final project, use classes that match the visual defects we can actually inspect:

- `Ready` → visually acceptable
- `Broken` → reject
- `Bent` → reject

**Under-dried is not a camera-only class here.** Drying status is better obtained from the ESP32 temperature/humidity logic. This avoids claiming that a normal camera can directly measure internal moisture.

In [ ]:
CLASS_NAMES = {
    0: "Ready",
    1: "Broken",
    2: "Bent"
}

print("YOLO classes:")
for k, v in CLASS_NAMES.items():
    print(f"{k}: {v}")

# 4. Dataset structure and class consistency

Your YOLO dataset must use exactly the same class order as above:

```text
Agarbatti_Dataset/
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── data.yaml
```

Every label must use:
- `0` = Ready
- `1` = Broken
- `2` = Bent

Do not train on width, length or colour as separate YOLO classes. Those are measurements/features handled by OpenCV.

In [ ]:
dataset_path = "Agarbatti_Dataset"

data_config = {
    "path": dataset_path,
    "train": "train/images",
    "val": "val/images",
    "names": CLASS_NAMES
}

with open("data.yaml", "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(open("data.yaml").read())

# 5. Train YOLOv8n

YOLOv8n is a sensible lightweight starting point for a laptop/Raspberry Pi-style prototype.

**Do not expect good real-world accuracy from synthetic/repeated images alone.** Use real photos of Ready, Broken and Bent agarbatti under different lighting, positions and batches.

A separate real test set should not be used for training.

In [ ]:
# Uncomment when your real labelled dataset is ready.

# model = YOLO("yolov8n.pt")
# model.train(
#     data="data.yaml",
#     epochs=50,
#     imgsz=640,
#     batch=8,
#     project="agarbatti_runs",
#     name="quality_grading",
#     patience=15
# )

print("Training cell ready.")

# 6. Input image / camera

For a prototype, either:
1. set `IMAGE_PATH` to a saved camera image, or
2. use the camera capture function below.

For the physical system, the camera is placed right after the drying stage — the agarbatti is placed manually (no conveyor is used).

In [ ]:
IMAGE_PATH = "agarbatti_image.jpg"

print("Image:", IMAGE_PATH)
print("Exists:", os.path.exists(IMAGE_PATH))

In [ ]:
def capture_agarbatti_image(output_file="agarbatti_image.jpg", camera_index=0):
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        print("ERROR: Camera could not be opened.")
        return None

    print("Press SPACE to capture; ESC to exit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("ERROR: Cannot read camera frame.")
            break

        cv2.imshow("Smart Agarbatti Inspection", frame)
        key = cv2.waitKey(1) & 0xFF

        if key == 32:
            cv2.imwrite(output_file, frame)
            cap.release()
            cv2.destroyAllWindows()
            print("Captured:", output_file)
            return output_file

        if key == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    return None

# Example:
# IMAGE_PATH = capture_agarbatti_image()

# 7. OpenCV preprocessing for visual inspection

We keep the **original image resolution** for measurement. We do not resize the image to 640×640 before measuring physical properties, because that would make pixel measurements less trustworthy.

YOLO can perform its own internal resizing/letterboxing during inference.

In [ ]:
def preprocess_for_opencv(image):
    if image is None:
        raise ValueError("Input image is empty.")

    denoised = cv2.GaussianBlur(image, (3, 3), 0)

    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_l = clahe.apply(l)

    enhanced = cv2.merge([enhanced_l, a, b])
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


def load_and_preprocess(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")
    return image, preprocess_for_opencv(image)


if os.path.exists(IMAGE_PATH):
    original, processed = load_and_preprocess(IMAGE_PATH)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
    plt.title("OpenCV enhanced")
    plt.axis("off")
    plt.show()

# 8. Load trained YOLOv8 model

Change `MODEL_PATH` to the location of your trained `best.pt`.

In [ ]:
MODEL_PATH = "agarbatti_runs/quality_grading/weights/best.pt"

if os.path.exists(MODEL_PATH):
    model = YOLO(MODEL_PATH)
    print("Custom YOLOv8 model loaded.")
else:
    model = None
    print("Custom model not found. Train the model first or update MODEL_PATH.")

# 9. YOLOv8 detection

YOLO gives us:
- the agarbatti bounding box
- the predicted class
- the confidence score

OpenCV then works **inside each YOLO box**.

In [ ]:
def run_yolo_detection(model, image_path, confidence=0.35):
    if model is None:
        raise ValueError("YOLO model is not loaded.")
    return model(image_path, conf=confidence, verbose=False)

# 10. OpenCV object measurement

The bounding box gives pixel dimensions.

For true millimetres, the camera must be calibrated using a known reference (for example, a ruler or ArUco marker) at the same inspection plane.

Without calibration, report pixels — do not pretend pixels are millimetres.

In [ ]:
def measure_detected_object(box):
    x1, y1, x2, y2 = map(int, box)

    width_px = max(0, x2 - x1)
    height_px = max(0, y2 - y1)

    long_side = max(width_px, height_px)
    short_side = max(1, min(width_px, height_px))

    return {
        "width_px": width_px,
        "height_px": height_px,
        "aspect_ratio": round(long_side / short_side, 2)
    }

# 11. Optional pixel-to-mm calibration

If the inspection camera is fixed and you know the number of pixels corresponding to 1 mm, set `PIXELS_PER_MM`.

Example: if 10 mm = 120 pixels, then `PIXELS_PER_MM = 12`.

If you do not have calibration yet, leave it as `None`.

In [ ]:
PIXELS_PER_MM = None  # Example: 12.0

def add_mm_measurements(features, pixels_per_mm=PIXELS_PER_MM):
    out = dict(features)

    if pixels_per_mm is not None and pixels_per_mm > 0:
        out["width_mm"] = round(out["width_px"] / pixels_per_mm, 2)
        out["height_mm"] = round(out["height_px"] / pixels_per_mm, 2)
    else:
        out["width_mm"] = None
        out["height_mm"] = None

    return out

# 12. OpenCV colour analysis

The raw YOLO crop contains background pixels. To avoid averaging the background into the colour result, we create a simple foreground mask.

This is a **supporting feature**, not the main defect detector.

In [ ]:
def make_foreground_mask(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu gives a starting mask for a contrasting inspection background.
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return mask


def extract_colour_features(crop):
    mask = make_foreground_mask(crop)

    if cv2.countNonZero(mask) < 20:
        # Safe fallback if segmentation is too small.
        mask = np.ones(mask.shape, dtype=np.uint8) * 255

    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    mean_bgr = cv2.mean(crop, mask=mask)[:3]
    mean_hsv = cv2.mean(hsv, mask=mask)[:3]

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    brightness = cv2.mean(gray, mask=mask)[0]

    return {
        "mean_blue": round(mean_bgr[0], 2),
        "mean_green": round(mean_bgr[1], 2),
        "mean_red": round(mean_bgr[2], 2),
        "mean_hue": round(mean_hsv[0], 2),
        "mean_saturation": round(mean_hsv[1], 2),
        "mean_value": round(mean_hsv[2], 2),
        "brightness": round(brightness, 2)
    }

# 13. Extract all YOLO + OpenCV features

This is the integration point:

```text
YOLO class/confidence/box
          +
OpenCV size
          +
OpenCV colour
          ↓
one feature record per agarbatti
```

In [ ]:
def extract_all_features(image_path, results):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(image_path)

    feature_records = []

    for result in results:
        names = result.names

        for box_obj in result.boxes:
            x1, y1, x2, y2 = map(int, box_obj.xyxy[0].tolist())

            confidence = float(box_obj.conf[0])
            class_id = int(box_obj.cls[0])
            class_name = names[class_id]

            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(image.shape[1], x2)
            y2 = min(image.shape[0], y2)

            crop = image[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            size_features = measure_detected_object([x1, y1, x2, y2])
            size_features = add_mm_measurements(size_features)

            colour_features = extract_colour_features(crop)

            feature_records.append({
                "class": class_name,
                "confidence": round(confidence * 100, 2),
                "box": [x1, y1, x2, y2],
                **size_features,
                **colour_features
            })

    return feature_records

# 14. Drying status from ESP32

The ESP32 controls the drying chamber and reads temperature/humidity.

For the notebook, we represent the ESP32 result with one variable:

- `READY_TO_INSPECT` → drying stage is complete
- `UNDER_DRYING` → keep drying; do not send to final quality grading yet
- `UNKNOWN` → manual check

In the real prototype, replace this variable with serial/Bluetooth/Wi-Fi data from the ESP32.

In [ ]:
DRYING_STATUS = "READY_TO_INSPECT"
# Possible values: "READY_TO_INSPECT", "UNDER_DRYING", "UNKNOWN"

def validate_drying_status(status):
    allowed = {"READY_TO_INSPECT", "UNDER_DRYING", "UNKNOWN"}
    if status not in allowed:
        raise ValueError(f"DRYING_STATUS must be one of {allowed}")
    return status

# 15. Final rule-based decision engine

This is the actual **decision-making layer**.

It combines:
1. YOLO defect result
2. Drying status from ESP32
3. Optional OpenCV dimensional limits
4. Optional colour limits

No second AI model is required.

### Example grading logic

- Broken → **REJECT**
- Bent → **REJECT**
- Still drying → **GRADE C / FURTHER DRY**
- Ready + measurements inside limits → **GRADE A**
- Ready but measurements outside limits → **GRADE C**
- Low-confidence/unknown → **MANUAL CHECK**

In [ ]:
# Set these only after measuring REAL good samples.
# Keep as None until you have calibrated project limits.

MIN_WIDTH_PX = None
MAX_WIDTH_PX = None
MIN_HEIGHT_PX = None
MAX_HEIGHT_PX = None

# Example brightness limits; calibrate from real samples before enabling.
MIN_BRIGHTNESS = None
MAX_BRIGHTNESS = None


def within_limits(value, low=None, high=None):
    if value is None:
        return True
    if low is not None and value < low:
        return False
    if high is not None and value > high:
        return False
    return True


def quality_decision(record, drying_status=DRYING_STATUS, min_confidence=50):
    validate_drying_status(drying_status)

    cls = record["class"]
    conf = record["confidence"]
    reasons = []

    # Safety gate: drying stage comes first.
    if drying_status == "UNDER_DRYING":
        return {
            "status": "UNDER-DRYING",
            "grade": "GRADE C",
            "action": "KEEP DRYING",
            "reason": "ESP32 drying status says the batch is not ready for inspection."
        }

    if drying_status == "UNKNOWN":
        return {
            "status": "UNKNOWN DRYING STATUS",
            "grade": "MANUAL CHECK",
            "action": "CHECK SENSORS",
            "reason": "Drying status is unavailable or invalid."
        }

    # Defect gate.
    if cls == "Broken":
        return {
            "status": "BROKEN",
            "grade": "REJECT",
            "action": "REMOVE",
            "reason": "YOLOv8 detected a broken agarbatti."
        }

    if cls == "Bent":
        return {
            "status": "BENT",
            "grade": "REJECT",
            "action": "REMOVE",
            "reason": "YOLOv8 detected a bent agarbatti."
        }

    if conf < min_confidence:
        return {
            "status": "LOW CONFIDENCE",
            "grade": "MANUAL CHECK",
            "action": "RECHECK IMAGE",
            "reason": f"YOLO confidence is only {conf:.1f}%."
        }

    # Measurement checks.
    if not within_limits(record["width_px"], MIN_WIDTH_PX, MAX_WIDTH_PX):
        reasons.append("width outside calibrated range")

    if not within_limits(record["height_px"], MIN_HEIGHT_PX, MAX_HEIGHT_PX):
        reasons.append("length/height outside calibrated range")

    if not within_limits(record["brightness"], MIN_BRIGHTNESS, MAX_BRIGHTNESS):
        reasons.append("brightness outside calibrated range")

    if reasons:
        return {
            "status": "READY BUT OUT OF SPEC",
            "grade": "GRADE C",
            "action": "DOWNGRADE / CHECK",
            "reason": "; ".join(reasons)
        }

    return {
        "status": "READY",
        "grade": "GRADE A",
        "action": "ACCEPT",
        "reason": "YOLO found Ready, drying is complete, and enabled OpenCV checks passed."
    }

# 16. Complete inspection pipeline

The order matters:

```text
IMAGE
  ↓
YOLO detects each agarbatti
  ↓
OpenCV measures each YOLO region
  ↓
ESP32 drying status is checked
  ↓
Decision engine combines everything
  ↓
Final grade + reason + action
```

In [ ]:
def complete_agarbatti_pipeline(image_path, model, drying_status=DRYING_STATUS):
    if model is None:
        raise ValueError("YOLO model is not loaded.")

    print("=" * 65)
    print("SMART AGARBATTI QUALITY INSPECTION")
    print("=" * 65)

    # YOLO handles its own inference resizing.
    results = run_yolo_detection(model, image_path)
    print("✓ YOLOv8 detection completed")

    records = extract_all_features(image_path, results)
    print(f"✓ OpenCV features extracted for {len(records)} object(s)")

    final_records = []

    for i, record in enumerate(records, start=1):
        decision = quality_decision(record, drying_status=drying_status)

        final_records.append({
            "Agarbatti_No": i,
            **record,
            **decision
        })

    return results, final_records

# 17. Run the system

This cell gives a table that can be shown to judges.

If there are no detections, first check:
- the model path
- camera/image quality
- YOLO confidence
- whether the dataset labels are correct

In [ ]:
if model is not None and os.path.exists(IMAGE_PATH):
    results, final_records = complete_agarbatti_pipeline(
        IMAGE_PATH, model, DRYING_STATUS
    )

    df = pd.DataFrame(final_records)
    display(df)
else:
    print("Provide IMAGE_PATH and a trained MODEL_PATH first.")

# 18. Visual result

The boxes and class names shown here are YOLO's visual detection output.

In [ ]:
if model is not None and os.path.exists(IMAGE_PATH):
    results = run_yolo_detection(model, IMAGE_PATH)
    annotated = results[0].plot()

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("YOLOv8 Agarbatti Detection")
    plt.axis("off")
    plt.show()

# 19. Batch summary

In [ ]:
def batch_summary(final_records):
    summary = {
        "total": len(final_records),
        "grade_a": 0,
        "grade_c": 0,
        "reject": 0,
        "manual_check": 0
    }

    for r in final_records:
        grade = r.get("grade", "")
        if grade == "GRADE A":
            summary["grade_a"] += 1
        elif grade == "GRADE C":
            summary["grade_c"] += 1
        elif grade == "REJECT":
            summary["reject"] += 1
        else:
            summary["manual_check"] += 1

    return summary


if "final_records" in globals():
    summary = batch_summary(final_records)

    print("\nBATCH QUALITY SUMMARY")
    print("=" * 35)
    print("Total inspected:", summary["total"])
    print("GRADE A:", summary["grade_a"])
    print("GRADE C:", summary["grade_c"])
    print("REJECT:", summary["reject"])
    print("MANUAL CHECK:", summary["manual_check"])

# 20. Final project architecture

```text
                    SOLAR PANEL
                         ↓
                      BATTERY
                         ↓
                       ESP32
                    ↙          ↘
          TEMP/HUMIDITY       FAN/ACTUATOR
                    ↓              ↓
                DRYING CHAMBER
                         ↓
                      CAMERA
                 (manual placement)
                         ↓
                  ORIGINAL IMAGE
                         ↓
                    YOLOv8
             Ready / Broken / Bent
                         ↓
                YOLO bounding boxes
                         ↓
                     OpenCV
          width / length / colour / brightness
                         ↓
                 FEATURE RECORD
                         +
                 ESP32 DRYING STATUS
                         ↓
              RULE-BASED DECISION ENGINE
                         ↓
          ┌──────────────┼──────────────┐
          ↓              ↓              ↓
       GRADE A         GRADE C        REJECT
       ACCEPT        KEEP/CHECK      REMOVE
```

## What each part does

**ESP32:** controls the drying chamber and reads temperature/humidity.

**YOLOv8:** looks at the camera image and recognizes visible quality conditions such as Ready, Broken and Bent.

**OpenCV:** directly measures image properties such as pixel width/height, aspect ratio, colour and brightness.

**Decision engine:** combines those outputs and gives the final grade and action.

## One sentence for judges

> “Our system uses ESP32-based smart drying, manual placement under a camera for inspection, YOLOv8 for visual defect detection, OpenCV for measurable quality features, and a rule-based decision engine to produce an explainable final grade.”

# 21. Important calibration checklist before the demo

1. Replace synthetic/repeated training images with real labelled photos wherever possible.
2. Make sure YOLO class IDs are exactly `0=Ready, 1=Broken, 2=Bent`.
3. Test on images the model never saw during training.
4. Fix the camera position and lighting.
5. If you claim millimetres, perform camera calibration using a known reference.
6. Measure real good samples before setting width/length/brightness limits.
7. If the ESP32 is the source of drying status, connect its serial/Bluetooth/Wi-Fi output to this notebook/application for the live demo.
8. Do not claim that the camera directly measures internal moisture.
9. Keep the final decision explainable: show the predicted class, confidence, measurements, drying status, final grade and reason.